# OneHealth-ML Full Pipeline
Verifies, harmonises, merges, labels, engineers, and splits all NHANES cycles
(2003–2018 for training, 2021–2023 for external validation).

In [10]:
import pandas as pd
import numpy as np
import os, sys, json, warnings
warnings.filterwarnings("ignore")

def find_project_root(marker="data"):
    here = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(here, marker)) and os.path.isdir(os.path.join(here, "src")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    raise RuntimeError("Could not locate project root")

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

RAW       = os.path.join(PROJECT_ROOT, "data", "raw")
INTERIM   = os.path.join(PROJECT_ROOT, "data", "interim")
PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(INTERIM, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

RANDOM_STATE = 42

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

print("Project root :", PROJECT_ROOT)
print("RAW          :", RAW)
print("RAW contents :", sorted(os.listdir(RAW)) if os.path.isdir(RAW) else "N/A")

Project root : c:\Mir\OneHealth-ML
RAW          : c:\Mir\OneHealth-ML\data\raw
RAW contents : ['NHANES_2005_2006', 'NHANES_2007_2008', 'NHANES_2009_2010', 'NHANES_2011_2012', 'NHANES_2013_2014', 'NHANES_2015_2016', 'NHANES_2017_2018', 'NHANES_2021_2023']


In [11]:
TRAINING_CYCLES = [
    ("NHANES_2005_2006", "_D", "2005-2006", 0),
    ("NHANES_2007_2008", "_E", "2007-2008", 1),
    ("NHANES_2009_2010", "_F", "2009-2010", 2),
    ("NHANES_2011_2012", "_G", "2011-2012", 3),
    ("NHANES_2013_2014", "_H", "2013-2014", 4),
    ("NHANES_2015_2016", "_I", "2015-2016", 5),
    ("NHANES_2017_2018", "_J", "2017-2018", 6),
]
EXTERNAL_CYCLE = ("NHANES_2021_2023", "_L", "2021-2023", 7)

COMPONENTS = ["DEMO","BMX","GHB","GLU","BPX","BIOPRO","DIQ","KIQ_U",
              "SMQ","ALQ","PAQ","DBQ","MCQ","BPQ","CBC","HDL",
              "TCHOL","TRIGLY","INS","HSCRP","ALB_CR","HSQ"]

# Filename overrides:
# - 2021-2023 uses BPXO instead of BPX
# - 2005-2010 cycles store HSCRP under CRP_* (variable LBXCRP)
# - CRP_G (2011-2012) and HSCRP_H (2013-2014) are NOT publicly available
FILE_OVERRIDE = {
    ("BPX",   "_L"): "BPXO_L.xpt",
    ("HSCRP", "_D"): "CRP_D.xpt",
    ("HSCRP", "_E"): "CRP_E.xpt",
    ("HSCRP", "_F"): "CRP_F.xpt",
    # No HSCRP for _G, _H — column will be NaN and imputed later
}

REQUIRED_COLS = {
    "DEMO":   ["SEQN","RIDSTATR","RIDAGEYR","RIAGENDR","RIDRETH3","INDFMPIR",
               "DMDEDUC2","DMDMARTL","RIDEXPRG","DMDBORN4","DMDYRSUS",
               "INDHHIN2","DMDHHSIZ","RIDEXMON","WTMEC2YR","SDMVPSU","SDMVSTRA"],
    "BMX":    ["SEQN","BMXWT","BMXHT","BMXBMI","BMXWAIST","BMXHIP",
               "BMXARMC","BMXARML","BMXLEG"],
    "GHB":    ["SEQN","LBXGH"],
    "GLU":    ["SEQN","LBXGLU","WTSAF2YR"],
    "BPX":    ["SEQN","BPXSY1","BPXDI1","BPXSY2","BPXDI2","BPXSY3","BPXDI3",
               "BPXSY4","BPXDI4","BPXPLS","BPXPULS"],
    "BIOPRO": ["SEQN","LBXSCR","LBXSBU","LBXSAL","LBXSTP","LBXSCA","LBXSNASI",
               "LBXSKSI","LBXSCLSI","LBXSTB","LBXSAPSI","LBXSATSI","LBXSASSI",
               "LBXSGTSI","LBXSPH","LBXSC3SI","LBXSUA","LBXSIR","LBXSGB",
               "LBXSCK","LBXSLDSI","LBXSOSSI"],
    "DIQ":    ["SEQN","DIQ010","DIQ160","DID040","DIQ080"],
    "KIQ_U":  ["SEQN","KIQ022","KIQ025","KIQ026"],
    "SMQ":    ["SEQN","SMQ020","SMD030","SMQ040","SMD650","SMD057",
               "SMQ890","SMQ900","SMQ910"],
    "ALQ":    ["SEQN","ALQ111","ALQ121","ALQ130","ALQ142","ALQ151"],
    "PAQ":    ["SEQN","PAQ605","PAQ620","PAQ635","PAQ650","PAQ665","PAD680"],
    "DBQ":    ["SEQN","DBQ700","DBQ197","DBD895","DBD900","DBD905","DBD910"],
    "MCQ":    ["SEQN","MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F",
               "MCQ160L","MCQ300C","MCQ300A"],
    "BPQ":    ["SEQN","BPQ020"],
    "CBC":    ["SEQN","LBXWBCSI","LBXHGB","LBXRDW","LBXPLTSI","LBXLYPCT","LBXNEPCT"],
    "HDL":    ["SEQN","LBDHDD"],
    "TCHOL":  ["SEQN","LBXTC"],
    "TRIGLY": ["SEQN","LBXTR"],
    "INS":    ["SEQN","LBXIN"],
    "HSCRP":  ["SEQN","LBXHSCRP"],
    "ALB_CR": ["SEQN","URDACT"],
    "HSQ":    ["SEQN","HSD010"],
}

In [12]:
def harmonise(df, suffix):
    # HSCRP: older cycles store as LBXCRP
    if "LBXCRP" in df.columns and "LBXHSCRP" not in df.columns:
        df = df.rename(columns={"LBXCRP": "LBXHSCRP"})

    # ALQ redesign: pre-2013-2014 uses ALQ101/120Q/141Q
    if suffix in ["_D","_E","_F","_G","_H","_I"]:
        alq_map = {}
        if "ALQ101"  in df.columns: alq_map["ALQ101"]  = "ALQ111"
        if "ALQ120Q" in df.columns: alq_map["ALQ120Q"] = "ALQ121"
        if "ALQ141Q" in df.columns: alq_map["ALQ141Q"] = "ALQ142"
        if alq_map:
            df = df.rename(columns=alq_map)

    # 2021-2023 uses BPXO and LBXTLG
    if suffix == "_L":
        bp_map = {
            "BPXOSY1":"BPXSY1","BPXODI1":"BPXDI1",
            "BPXOSY2":"BPXSY2","BPXODI2":"BPXDI2",
            "BPXOSY3":"BPXSY3","BPXODI3":"BPXDI3",
            "BPXOSY4":"BPXSY4","BPXODI4":"BPXDI4",
            "BPXOPLS":"BPXPLS","BPXOPULS":"BPXPULS",
        }
        df = df.rename(columns={k:v for k,v in bp_map.items() if k in df.columns})
        if "LBXTLG" in df.columns and "LBXTR" not in df.columns:
            df = df.rename(columns={"LBXTLG": "LBXTR"})

    return df

In [13]:
import os

RAW = os.path.join(PROJECT_ROOT, "data", "raw")
bad_files = []
good_count = 0

for folder in sorted(os.listdir(RAW)):
    folder_path = os.path.join(RAW, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if not fname.endswith(".xpt"):
            continue
        fpath = os.path.join(folder_path, fname)
        size = os.path.getsize(fpath)
        # Read first 100 bytes to check for XPORT header
        with open(fpath, "rb") as f:
            head = f.read(100)
        # Real .xpt files start with "HEADER RECORD*******LIBRARY HEADER RECORD"
        is_xport = b"HEADER RECORD" in head[:80]
        is_html = b"<html" in head.lower() or b"<!doctype" in head.lower()
        if not is_xport or is_html or size < 5000:
            bad_files.append((folder, fname, size, head[:60]))
        else:
            good_count += 1

print(f"Good files: {good_count}")
print(f"Bad files : {len(bad_files)}")
print()
for folder, fname, size, head in bad_files:
    print(f"  ❌ {folder}/{fname}  ({size} bytes)")
    print(f"     head: {head}")

Good files: 174
Bad files : 0



In [14]:
missing_report = {}
loaded_per_cycle = {}

for folder, suffix, label, code in TRAINING_CYCLES + [EXTERNAL_CYCLE]:
    folder_path = os.path.join(RAW, folder)
    if not os.path.isdir(folder_path):
        print(f"❌ {folder} not found, skipping.")
        continue

    print(f"\n=== {label} ({suffix}) ===")
    cycle_frames = {}

    for comp in COMPONENTS:
        fname = FILE_OVERRIDE.get((comp, suffix), f"{comp}{suffix}.xpt")
        fpath = os.path.join(folder_path, fname)
        if not os.path.exists(fpath):
            # Expected for HSCRP in _G and _H; silently skip
            if comp == "HSCRP":
                continue
            print(f"  {comp}: file missing ({fname})")
            continue

        try:
            df = pd.read_sas(fpath)
        except Exception as e:
            print(f"  {comp}: read failed ({fname}): {e}")
            continue

        df = harmonise(df, suffix)
        df["cycle"] = code

        keep = [c for c in REQUIRED_COLS[comp] if c in df.columns]
        if len(keep) == 0:
            print(f"  {comp}: no usable columns")
            continue
        df = df[keep].copy()

        missing = [c for c in REQUIRED_COLS[comp] if c not in df.columns]
        if missing:
            missing_report[(label, comp)] = missing

        cycle_frames[comp] = df

    loaded_per_cycle[label] = cycle_frames
    print(f"  Loaded {len(cycle_frames)} components")

print(f"\n=== Missing variable report ===")
if missing_report:
    for (lab, comp), miss in sorted(missing_report.items()):
        print(f"{lab} {comp}: {miss}")
else:
    print("No unexpected missing variables.")

print("\nAll cycles loaded.")


=== 2005-2006 (_D) ===
  Loaded 22 components

=== 2007-2008 (_E) ===
  Loaded 22 components

=== 2009-2010 (_F) ===
  Loaded 22 components

=== 2011-2012 (_G) ===
  Loaded 21 components

=== 2013-2014 (_H) ===
  Loaded 21 components

=== 2015-2016 (_I) ===
  Loaded 22 components

=== 2017-2018 (_J) ===
  Loaded 22 components

=== 2021-2023 (_L) ===
  Loaded 22 components

=== Missing variable report ===
2005-2006 ALB_CR: ['URDACT']
2005-2006 ALQ: ['ALQ142', 'ALQ151']
2005-2006 BIOPRO: ['LBXSCK']
2005-2006 BMX: ['BMXHIP']
2005-2006 DBQ: ['DBD895', 'DBD900', 'DBD905', 'DBD910']
2005-2006 DEMO: ['RIDRETH3', 'DMDBORN4', 'INDHHIN2']
2005-2006 KIQ_U: ['KIQ026']
2005-2006 PAQ: ['PAQ605', 'PAQ620', 'PAQ635', 'PAQ650', 'PAQ665', 'PAD680']
2005-2006 SMQ: ['SMQ890', 'SMQ900', 'SMQ910']
2007-2008 ALB_CR: ['URDACT']
2007-2008 ALQ: ['ALQ142', 'ALQ151']
2007-2008 BIOPRO: ['LBXSCK']
2007-2008 BMX: ['BMXHIP']
2007-2008 DEMO: ['RIDRETH3', 'DMDBORN4']
2007-2008 SMQ: ['SMQ890', 'SMQ900', 'SMQ910']
2009-

In [15]:
missing_report = {}
loaded_per_cycle = {}

for folder, suffix, label, code in TRAINING_CYCLES + [EXTERNAL_CYCLE]:
    folder_path = os.path.join(RAW, folder)
    if not os.path.isdir(folder_path):
        print(f"❌ {folder} not found, skipping.")
        continue

    print(f"\n=== {label} ({suffix}) ===")
    cycle_frames = {}

    for comp in COMPONENTS:
        fname = FILE_OVERRIDE.get((comp, suffix), f"{comp}{suffix}.xpt")
        fpath = os.path.join(folder_path, fname)
        if not os.path.exists(fpath):
            if comp == "HSCRP":
                continue
            print(f"  {comp}: file missing ({fname})")
            continue

        try:
            df = pd.read_sas(fpath)
        except Exception as e:
            print(f"  {comp}: read failed ({fname}): {e}")
            continue

        df = harmonise(df, suffix)
        df["cycle"] = code

        # ALWAYS keep SEQN and cycle, plus any required columns that exist
        keep = ["SEQN", "cycle"] + [c for c in REQUIRED_COLS[comp] if c in df.columns and c not in ("SEQN", "cycle")]
        df = df[keep].copy()

        missing = [c for c in REQUIRED_COLS[comp] if c not in df.columns and c not in ("SEQN", "cycle")]
        if missing:
            missing_report[(label, comp)] = missing

        cycle_frames[comp] = df

    loaded_per_cycle[label] = cycle_frames
    print(f"  Loaded {len(cycle_frames)} components")

print(f"\n=== Missing variable report ===")
if missing_report:
    for (lab, comp), miss in sorted(missing_report.items()):
        print(f"{lab} {comp}: {miss}")
else:
    print("No unexpected missing variables.")

print("\nAll cycles loaded.")


=== 2005-2006 (_D) ===
  Loaded 22 components

=== 2007-2008 (_E) ===
  Loaded 22 components

=== 2009-2010 (_F) ===
  Loaded 22 components

=== 2011-2012 (_G) ===
  Loaded 21 components

=== 2013-2014 (_H) ===
  Loaded 21 components

=== 2015-2016 (_I) ===
  Loaded 22 components

=== 2017-2018 (_J) ===
  Loaded 22 components

=== 2021-2023 (_L) ===
  Loaded 22 components

=== Missing variable report ===
2005-2006 ALB_CR: ['URDACT']
2005-2006 ALQ: ['ALQ142', 'ALQ151']
2005-2006 BIOPRO: ['LBXSCK']
2005-2006 BMX: ['BMXHIP']
2005-2006 DBQ: ['DBD895', 'DBD900', 'DBD905', 'DBD910']
2005-2006 DEMO: ['RIDRETH3', 'DMDBORN4', 'INDHHIN2']
2005-2006 KIQ_U: ['KIQ026']
2005-2006 PAQ: ['PAQ605', 'PAQ620', 'PAQ635', 'PAQ650', 'PAQ665', 'PAD680']
2005-2006 SMQ: ['SMQ890', 'SMQ900', 'SMQ910']
2007-2008 ALB_CR: ['URDACT']
2007-2008 ALQ: ['ALQ142', 'ALQ151']
2007-2008 BIOPRO: ['LBXSCK']
2007-2008 BMX: ['BMXHIP']
2007-2008 DEMO: ['RIDRETH3', 'DMDBORN4']
2007-2008 SMQ: ['SMQ890', 'SMQ900', 'SMQ910']
2009-

In [16]:
training_frames = []

for folder, suffix, label, code in TRAINING_CYCLES:
    if label not in loaded_per_cycle:
        continue
    cycle_frames = loaded_per_cycle[label]
    if "DEMO" not in cycle_frames:
        print(f"{label}: DEMO missing, skipping")
        continue

    merged = cycle_frames["DEMO"].copy()
    for comp in COMPONENTS:
        if comp == "DEMO" or comp not in cycle_frames:
            continue
        right = cycle_frames[comp]
        merged = merged.merge(right, on=["SEQN", "cycle"], how="inner")
    training_frames.append(merged)
    print(f"{label}: merged {merged.shape}")

if not training_frames:
    raise RuntimeError("No training frames loaded")

train_master = pd.concat(training_frames, ignore_index=True, sort=False)
print(f"\nCombined training shape: {train_master.shape}")
print("Duplicate SEQN+cycle:", train_master[["SEQN", "cycle"]].duplicated().sum())

2005-2006: merged (2289, 92)
2007-2008: merged (2776, 104)
2009-2010: merged (2926, 105)
2011-2012: merged (2627, 109)
2013-2014: merged (2668, 109)
2015-2016: merged (2598, 113)
2017-2018: merged (2541, 114)

Combined training shape: (18425, 114)
Duplicate SEQN+cycle: 0


In [18]:
def egfr_ckdepi_2021(scr, age, sex):
    scr = np.asarray(scr, dtype=float)
    age = np.asarray(age, dtype=float)
    sex = np.asarray(sex, dtype=float)
    kappa = np.where(sex == 1, 0.9, 0.7)
    alpha = np.where(sex == 1, -0.302, -0.241)
    ratio = scr / kappa
    rmin = np.minimum(ratio, 1)
    rmax = np.maximum(ratio, 1)
    return 142 * (rmin ** alpha) * (rmax ** -1.200) * (0.9938 ** age) * np.where(sex == 1, 1.0, 1.012)


def add_derived_means(df):
    bp_sys = [c for c in ["BPXSY1","BPXSY2","BPXSY3","BPXSY4"] if c in df.columns]
    bp_dia = [c for c in ["BPXDI1","BPXDI2","BPXDI3","BPXDI4"] if c in df.columns]
    df["mean_sbp"] = df[bp_sys].mean(axis=1, skipna=True) if bp_sys else np.nan
    df["mean_dbp"] = df[bp_dia].mean(axis=1, skipna=True) if bp_dia else np.nan
    df["eGFR"] = egfr_ckdepi_2021(df["LBXSCR"], df["RIDAGEYR"], df["RIAGENDR"]) if "LBXSCR" in df.columns else np.nan
    return df


def label_diabetes(r):
    if r.get("RIDEXPRG") == 1: return np.nan
    if pd.isna(r.get("LBXGH")) and pd.isna(r.get("LBXGLU")) and pd.isna(r.get("DIQ010")): return np.nan
    if (r.get("LBXGH", 0) >= 6.5) or (r.get("LBXGLU", 0) >= 126) or (r.get("DIQ010") == 1): return 1
    return 0

def label_hypertension(r):
    if pd.isna(r.get("mean_sbp")) and pd.isna(r.get("mean_dbp")) and pd.isna(r.get("BPQ020")): return np.nan
    if (r.get("mean_sbp", 0) >= 130) or (r.get("mean_dbp", 0) >= 80) or (r.get("BPQ020") == 1): return 1
    return 0

def label_ckd(r):
    if pd.isna(r.get("eGFR")) and pd.isna(r.get("KIQ025")) and pd.isna(r.get("KIQ022")): return np.nan
    if (r.get("eGFR", 100) < 60) or (r.get("KIQ025") == 1) or (r.get("KIQ022") == 1): return 1
    return 0

def label_dyslipidemia(r):
    if pd.isna(r.get("LBXTC")) and pd.isna(r.get("LBDHDD")) and pd.isna(r.get("LBXTR")): return np.nan
    hdl_low = (r.get("LBDHDD", 100) < 40) if r.get("RIAGENDR") == 1 else (r.get("LBDHDD", 100) < 50)
    if (r.get("LBXTC", 0) >= 200) or hdl_low or (r.get("LBXTR", 0) >= 150): return 1
    return 0

def label_cvd(r):
    vals = [r.get(c) for c in ["MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F"]]
    if all(pd.isna(v) for v in vals): return np.nan
    return 1 if any(v == 1 for v in vals) else 0

def label_metabolic_syndrome(r):
    comp = 0
    if (r.get("RIAGENDR") == 1 and r.get("BMXWAIST", 0) > 102) or (r.get("RIAGENDR") == 2 and r.get("BMXWAIST", 0) > 88):
        comp += 1
    if r.get("LBXTR", 0) >= 150: comp += 1
    hdl_low = (r.get("LBDHDD", 100) < 40) if r.get("RIAGENDR") == 1 else (r.get("LBDHDD", 100) < 50)
    if hdl_low: comp += 1
    if pd.notna(r.get("mean_sbp")) and pd.notna(r.get("mean_dbp")):
        if r["mean_sbp"] >= 130 or r["mean_dbp"] >= 85: comp += 1
    if r.get("LBXGLU", 0) >= 100: comp += 1
    if any(pd.isna(r.get(c)) for c in ["BMXWAIST","LBXTR","LBDHDD","mean_sbp","LBXGLU"]): return np.nan
    return 1 if comp >= 3 else 0

def label_liver_disease(r):
    if pd.isna(r.get("LBXSATSI")) and pd.isna(r.get("LBXSASSI")) and pd.isna(r.get("LBXSGTSI")): return np.nan
    alt_uln = 40 if r.get("RIAGENDR") == 1 else 30
    elevated = (r.get("LBXSATSI", 0) > alt_uln) or (r.get("LBXSASSI", 0) > 40) or (r.get("LBXSGTSI", 0) > 50)
    bmi_high = r.get("BMXBMI", 0) >= 25
    return 1 if (elevated and bmi_high) else 0


def generate_labels(df):
    df = add_derived_means(df)
    df["label_diabetes"]           = df.apply(label_diabetes, axis=1)
    df["label_hypertension"]       = df.apply(label_hypertension, axis=1)
    df["label_ckd"]                = df.apply(label_ckd, axis=1)
    df["label_dyslipidemia"]       = df.apply(label_dyslipidemia, axis=1)
    df["label_cvd"]                = df.apply(label_cvd, axis=1)
    df["label_metabolic_syndrome"] = df.apply(label_metabolic_syndrome, axis=1)
    df["label_liver_disease"]      = df.apply(label_liver_disease, axis=1)
    return df


train_master = generate_labels(train_master)
print(f"\nLabeled training shape: {train_master.shape}")
for c in [c for c in train_master.columns if c.startswith("label_")]:
    pos = int(train_master[c].sum())
    tot = int(train_master[c].notna().sum())
    print(f"  {c}: {pos}/{tot} positive ({100*pos/max(tot,1):.1f}%)")

train_master.to_csv(os.path.join(PROCESSED, "labeled_master_all_cycles.csv"), index=False)
print("Saved labeled_master_all_cycles.csv")

if ext is not None:
    ext = generate_labels(ext)
    ext.to_csv(os.path.join(PROCESSED, "labeled_external_2021_2023.csv"), index=False)
    print(f"External labeled shape: {ext.shape}")


Labeled training shape: (18425, 124)
  label_diabetes: 3459/18092 positive (19.1%)
  label_hypertension: 9688/18425 positive (52.6%)
  label_ckd: 1683/18424 positive (9.1%)
  label_dyslipidemia: 10870/17367 positive (62.6%)
  label_cvd: 2114/18424 positive (11.5%)
  label_metabolic_syndrome: 5462/15924 positive (34.3%)
  label_liver_disease: 2566/17307 positive (14.8%)
Saved labeled_master_all_cycles.csv
External labeled shape: (3425, 96)


In [19]:
SELECTED_FEATURES = [
    "RIDAGEYR","RIAGENDR","RIDRETH3","INDFMPIR","DMDEDUC2","DMDMARTL",
    "DMDBORN4","DMDYRSUS","INDHHIN2","DMDHHSIZ","RIDEXMON",
    "BMXWT","BMXHT","BMXBMI","BMXWAIST","BMXHIP","BMXARMC","BMXARML","BMXLEG",
    "LBXGH","LBXGLU",
    "BPXSY1","BPXDI1","BPXSY2","BPXDI2","BPXSY3","BPXDI3","BPXSY4","BPXDI4",
    "BPXPLS","BPXPULS",
    "LBXSCR","LBXSBU","LBXSAL","LBXSTP","LBXSCA","LBXSNASI","LBXSKSI",
    "LBXSCLSI","LBXSTB","LBXSAPSI","LBXSATSI","LBXSASSI","LBXSGTSI",
    "LBXSPH","LBXSC3SI","LBXSUA","LBXSIR","LBXSGB","LBXSCK","LBXSLDSI",
    "LBXSOSSI",
    "DIQ160","DID040","DIQ080",
    "KIQ022","KIQ025","KIQ026",
    "SMQ020","SMD030","SMQ040","SMD650","SMD057","SMQ890","SMQ900","SMQ910",
    "ALQ111","ALQ121","ALQ130","ALQ142","ALQ151",
    "PAQ605","PAQ620","PAQ635","PAQ650","PAQ665","PAD680",
    "DBQ700","DBQ197","DBD895","DBD900","DBD905","DBD910",
    "MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F",
    "MCQ160L","MCQ300C","MCQ300A",
    "LBXWBCSI","LBXHGB","LBXRDW","LBXPLTSI","LBXLYPCT","LBXNEPCT",
    "LBDHDD","LBXTC","LBXTR",
    "LBXIN","LBXHSCRP","URDACT",
    "HSD010",
]
ADMIN_VARS   = ["SEQN","WTMEC2YR","SDMVPSU","SDMVSTRA","RIDSTATR","BMDSTATS","WTSAF2YR"]
TEMP_SOURCES = ["DIQ010","BPQ020","mean_sbp","mean_dbp","eGFR"]
DERIVED      = ["NLR","HOMA_IR","WHR","ASCVD_risk","met_syndrome_count",
                "eGFR_stage","obesity_class","pack_years"]

LEAKAGE_MAP = {
    "label_diabetes": ["LBXGH","LBXGLU","DIQ010","HOMA_IR","LBXSGL",
                       "met_syndrome_count","ASCVD_risk","DIQ160","DID040","DIQ080"],
    "label_hypertension": ["BPXSY1","BPXSY2","BPXSY3","BPXSY4",
                           "BPXDI1","BPXDI2","BPXDI3","BPXDI4",
                           "BPXPLS","BPXPULS","BPQ020",
                           "mean_sbp","mean_dbp","ASCVD_risk"],
    "label_ckd": ["LBXSCR","eGFR","KIQ022","KIQ025","URDACT","eGFR_stage"],
    "label_dyslipidemia": ["LBXTC","LBDHDD","LBXTR","LBXSCH","LBXSTR","ASCVD_risk"],
    "label_cvd": ["MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F",
                  "MCQ300A","MCQ300B","MCQ300C"],
    "label_metabolic_syndrome": ["BMXWAIST","LBXTR","LBDHDD",
                                 "BPXSY1","BPXSY2","BPXSY3","BPXSY4",
                                 "BPXDI1","BPXDI2","BPXDI3","BPXDI4",
                                 "mean_sbp","mean_dbp","LBXGLU","WHR",
                                 "LBXSGL","LBXSTR","met_syndrome_count","ASCVD_risk"],
    "label_liver_disease": ["LBXSATSI","LBXSASSI","LBXSGTSI","BMXBMI","obesity_class"],
}


def add_derived_features(df):
    # NLR
    ly = df["LBXLYPCT"] if "LBXLYPCT" in df.columns else pd.Series(np.nan, index=df.index)
    ne = df["LBXNEPCT"] if "LBXNEPCT" in df.columns else pd.Series(np.nan, index=df.index)
    df["NLR"] = np.where(ly > 0, ne / ly, np.nan)

    # HOMA-IR
    ins = df["LBXIN"] if "LBXIN" in df.columns else pd.Series(np.nan, index=df.index)
    glu = df["LBXGLU"] if "LBXGLU" in df.columns else pd.Series(np.nan, index=df.index)
    df["HOMA_IR"] = (ins * glu) / 405

    # WHR
    if "BMXHIP" in df.columns and "BMXWAIST" in df.columns:
        df["WHR"] = df["BMXWAIST"] / df["BMXHIP"].replace(0, np.nan)
    else:
        df["WHR"] = np.nan

    # ASCVD risk
    smoker     = df["SMQ040"].isin([1,2]).astype(int) if "SMQ040" in df.columns else 0
    diabetic   = (df["DIQ010"] == 1).astype(int) if "DIQ010" in df.columns else 0
    ht_treated = (df["BPQ020"] == 1).astype(int) if "BPQ020" in df.columns else 0
    tc  = df["LBXTC"]  if "LBXTC"  in df.columns else 0
    hdl = df["LBDHDD"] if "LBDHDD" in df.columns else 0
    sbp = df["mean_sbp"] if "mean_sbp" in df.columns else 0
    df["ASCVD_risk"] = (
        df["RIDAGEYR"]*0.05 + tc*0.002 - hdl*0.003 + sbp*0.004
        + smoker*0.15 + diabetic*0.2 + ht_treated*0.1
    ).clip(0,1)

    # Metabolic syndrome count
    waist = df["BMXWAIST"] if "BMXWAIST" in df.columns else pd.Series(np.nan, index=df.index)
    tg    = df["LBXTR"]    if "LBXTR"    in df.columns else pd.Series(np.nan, index=df.index)
    hdl_s = df["LBDHDD"]   if "LBDHDD"   in df.columns else pd.Series(np.nan, index=df.index)
    sbp_s = df["mean_sbp"] if "mean_sbp" in df.columns else pd.Series(np.nan, index=df.index)
    dbp_s = df["mean_dbp"] if "mean_dbp" in df.columns else pd.Series(np.nan, index=df.index)
    glu_s = df["LBXGLU"]   if "LBXGLU"   in df.columns else pd.Series(np.nan, index=df.index)
    df["met_syndrome_count"] = (
        (((df["RIAGENDR"]==1)&(waist>102)) | ((df["RIAGENDR"]==2)&(waist>88))).astype(int)
        + (tg>=150).astype(int)
        + (((df["RIAGENDR"]==1)&(hdl_s<40)) | ((df["RIAGENDR"]==2)&(hdl_s<50))).astype(int)
        + ((sbp_s>=130)|(dbp_s>=85)).astype(int)
        + (glu_s>=100).astype(int)
    )

    # eGFR stage
    def stage(e):
        if pd.isna(e): return np.nan
        return 1 if e>=90 else 2 if e>=60 else 3 if e>=45 else 4 if e>=30 else 5
    df["eGFR_stage"] = df["eGFR"].apply(stage) if "eGFR" in df.columns else np.nan

    # Obesity class
    def obese(b):
        if pd.isna(b): return np.nan
        return 0 if b<18.5 else 1 if b<25 else 2 if b<30 else 3 if b<35 else 4 if b<40 else 5
    df["obesity_class"] = df["BMXBMI"].apply(obese) if "BMXBMI" in df.columns else np.nan

    # Pack-years
    def py(r):
        if r.get("SMQ020") != 1: return 0
        s = r.get("SMD030"); c = r.get("SMD650", 0)
        if pd.isna(s) or s <= 0: return 0
        c = 0 if pd.isna(c) else c
        return (c * max(0, r["RIDAGEYR"] - s)) / 20
    df["pack_years"] = df.apply(py, axis=1)
    return df


def curate(df):
    cols = SELECTED_FEATURES + TEMP_SOURCES + ADMIN_VARS
    cols += [c for c in df.columns if c.startswith("label_")]
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()


train_master = add_derived_features(train_master)
train_master = curate(train_master)
print(f"After curation: {train_master.shape}")

if ext is not None:
    ext = add_derived_features(ext)
    ext = curate(ext)
    print(f"External after curation: {ext.shape}")

After curation: (18425, 122)
External after curation: (3425, 94)


In [20]:
label_cols = [c for c in train_master.columns if c.startswith("label_")]
all_predictors = SELECTED_FEATURES + DERIVED
feature_names_json = {}

for label_col in label_cols:
    disease = label_col.replace("label_", "")
    mask = train_master[label_col].notna()
    y = train_master.loc[mask, label_col].astype(int)
    exclude = set(LEAKAGE_MAP.get(label_col, []))
    feats = [c for c in all_predictors if c not in exclude and c in train_master.columns]

    X = train_master.loc[mask, feats].copy()
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    X = X.dropna(axis=1, how="all")

    imp = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_imp, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    np.save(os.path.join(PROCESSED, f"{disease}_X_train.npy"), X_tr_s)
    np.save(os.path.join(PROCESSED, f"{disease}_X_test.npy"),  X_te_s)
    np.save(os.path.join(PROCESSED, f"{disease}_y_train.npy"), y_tr)
    np.save(os.path.join(PROCESSED, f"{disease}_y_test.npy"),  y_te)
    joblib.dump(scaler, os.path.join(PROCESSED, f"{disease}_scaler.pkl"))

    feature_names_json[label_col] = X.columns.tolist()
    print(f"{disease:22s} train={X_tr_s.shape} test={X_te_s.shape} "
          f"pos_train={y_tr.sum()} pos_test={y_te.sum()}")

with open(os.path.join(PROCESSED, "feature_names.json"), "w") as f:
    json.dump(feature_names_json, f, indent=2)
with open(os.path.join(PROCESSED, "leakage_map.json"), "w") as f:
    json.dump(LEAKAGE_MAP, f, indent=2)

print("\nSaved all arrays, scalers, and JSON metadata.")

if ext is not None:
    ext.to_csv(os.path.join(PROCESSED, "external_full_curated.csv"), index=False)
    print(f"External saved: {ext.shape}")

diabetes               train=(14473, 99) test=(3619, 99) pos_train=2767 pos_test=692
hypertension           train=(14740, 94) test=(3685, 94) pos_train=7750 pos_test=1938
ckd                    train=(14739, 100) test=(3685, 100) pos_train=1346 pos_test=337
dyslipidemia           train=(13893, 101) test=(3474, 101) pos_train=8696 pos_test=2174
cvd                    train=(14739, 97) test=(3685, 97) pos_train=1691 pos_test=423
metabolic_syndrome     train=(12739, 92) test=(3185, 92) pos_train=4370 pos_test=1092
liver_disease          train=(13845, 100) test=(3462, 100) pos_train=2053 pos_test=513

Saved all arrays, scalers, and JSON metadata.
External saved: (3425, 94)


In [21]:
print("\n" + "="*70)
print("PIPELINE COMPLETE")
print("="*70)
print(f"Training master  : {train_master.shape}")
print(f"External master  : {ext.shape if ext is not None else 'N/A'}")
print(f"Outputs in       : {PROCESSED}")
print("\nNext: upload data/processed/ to Google Drive and open the Colab deep-learning notebook.")


PIPELINE COMPLETE
Training master  : (18425, 122)
External master  : (3425, 94)
Outputs in       : c:\Mir\OneHealth-ML\data\processed

Next: upload data/processed/ to Google Drive and open the Colab deep-learning notebook.
